# Notebook 01: Exploratory Data Analysis (EDA), Preprocessing & Leakage-Free Pipeline

**Project**: Student Performance Classification using Classical Machine Learning  
**Author**: Raghav (Core ML Lead)  
**Objective**: Perform comprehensive EDA, inspect feature distributions and relationships, handle missing values via median imputation, configure feature scaling, and construct a leakage-free Scikit-Learn preprocessing pipeline.

> **Important Rule**: Raw data in `data/student_performance_data.csv` is preserved untouched. No ML model classifiers are trained in this notebook.

## 1. Environment Setup & Library Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Set aesthetic styling for academic plots
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11

## 2. Load Raw Dataset & Initial Inspection

In [ ]:
data_path = os.path.join('..', 'data', 'student_performance_data.csv')
if not os.path.exists(data_path):
    data_path = os.path.join('data', 'student_performance_data.csv')

df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 3. Feature & Target Selection (Preventing Data Leakage)

### Column Role Decisions:
1. **Input Features ($X$)**:
   - `MST_Score`: Mid-Semester examination score ($0-100$).
   - `Quiz_Score`: Continuous quiz assessment marks ($0-100$).
   - `Attendance_Percent`: Lecture attendance rate ($0-100\%$).
   - `Assignment_Score`: Practical homework marks ($0-100$).
2. **Target Label ($y$)**:
   - `Performance_Category`: 3 tiers (`Needs Improvement`, `Average Performer`, `High Performer`).
3. **Explicitly Excluded Columns**:
   - `Student_ID`: Identifier column with zero predictive generalization.
   - `Performance_Score`: **Direct Target Leakage**. As empirically proven in Phase 2A, `Performance_Score` is an exact weighted sum ($0.40\times\text{MST} + 0.20\times\text{Quiz} + 0.20\times\text{Att} + 0.20\times\text{Assign}$) that directly dictates the target thresholds. Including it would bypass learning real academic patterns.

In [ ]:
feature_cols = ['MST_Score', 'Quiz_Score', 'Attendance_Percent', 'Assignment_Score']
target_col = 'Performance_Category'

X = df[feature_cols].copy()
y = df[target_col].copy()

print("Feature matrix X shape:", X.shape)
print("Target vector y shape:", y.shape)

## 4. Train / Test Split (80% Train, 20% Test)

> **Why split BEFORE fitting preprocessing?**  
> Splitting before preprocessing is crucial to prevent **data leakage**. If an imputer or scaler computes statistics (like mean or median) across the entire dataset before splitting, information from the test set leaks into the training set. Therefore, the split is performed first, and all transformers are fitted strictly on $X_{train}$.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training set: {X_train.shape[0]} samples (80%)")
print(f"Testing set:  {X_test.shape[0]} samples (20%)")

# Verify class stratification proportions
strat_check = pd.DataFrame({
    'Full_Dataset (%)': (y.value_counts(normalize=True) * 100).round(2),
    'Train_Set (%)': (y_train.value_counts(normalize=True) * 100).round(2),
    'Test_Set (%)': (y_test.value_counts(normalize=True) * 100).round(2)
})
strat_check

## 5. Exploratory Data Analysis (EDA) & Visualizations

### 5.1 Descriptive Statistics for Input Features

In [ ]:
desc_df = pd.DataFrame({
    'Count': X.count(),
    'Mean': X.mean().round(4),
    'Std_Dev': X.std().round(4),
    'Min': X.min(),
    '25% (Q1)': X.quantile(0.25),
    'Median (Q2)': X.median(),
    '75% (Q3)': X.quantile(0.75),
    'Max': X.max(),
    'Missing_Values': X.isnull().sum()
})
desc_df

### 5.2 Target Class Distribution (Imbalance Inspection)

In [ ]:
counts = y.value_counts()
percentages = (y.value_counts(normalize=True) * 100).round(2)

plt.figure(figsize=(8, 5))
palette = {'Average Performer': '#3498db', 'High Performer': '#2ecc71', 'Needs Improvement': '#e74c3c'}
bars = plt.bar(counts.index, counts.values, color=[palette.get(c, '#555555') for c in counts.index], edgecolor='black', width=0.55)

for bar, count, pct in zip(bars, counts.values, percentages):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 10, f'{count} ({pct:.1f}%)', ha='center', va='bottom', fontweight='bold')

plt.title('Target Class Distribution (Performance Category)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Performance Category', fontsize=12)
plt.ylabel('Number of Students', fontsize=12)
plt.ylim(0, max(counts.values) + 80)
plt.tight_layout()
plt.show()

### 5.3 Univariate Distributions of Input Features

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = ['#2980b9', '#8e44ad', '#16a085', '#d35400']

for idx, col in enumerate(feature_cols):
    r, c = idx // 2, idx % 2
    sns.histplot(df[col].dropna(), kde=True, color=colors[idx], bins=25, ax=axes[r, c], edgecolor='black')
    axes[r, c].axvline(df[col].mean(), color='#e74c3c', linestyle='--', linewidth=2, label=f'Mean: {df[col].mean():.2f}')
    axes[r, c].axvline(df[col].median(), color='#27ae60', linestyle='-', linewidth=2, label=f'Median: {df[col].median():.2f}')
    axes[r, c].set_title(f'Distribution of {col}', fontsize=12, fontweight='bold')
    axes[r, c].set_xlabel(col, fontsize=11)
    axes[r, c].set_ylabel('Student Count', fontsize=11)
    axes[r, c].legend(frameon=True)

plt.suptitle('Histograms & Density Estimates of Academic Input Features', fontsize=15, fontweight='bold', y=0.99)
plt.tight_layout()
plt.show()

### 5.4 Feature Distributions Grouped by Student Performance Category

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
order = ['Needs Improvement', 'Average Performer', 'High Performer']
cat_palette = {'Needs Improvement': '#e74c3c', 'Average Performer': '#3498db', 'High Performer': '#2ecc71'}

for idx, col in enumerate(feature_cols):
    r, c = idx // 2, idx % 2
    sns.boxplot(data=df, x='Performance_Category', y=col, order=order, palette=cat_palette, ax=axes[r, c], width=0.5, hue='Performance_Category', legend=False)
    axes[r, c].set_title(f'{col} by Performance Tier', fontsize=12, fontweight='bold')
    axes[r, c].set_xlabel('Performance Category', fontsize=11)
    axes[r, c].set_ylabel(col, fontsize=11)

plt.suptitle('Bivariate Analysis: Features vs Performance Category', fontsize=15, fontweight='bold', y=0.99)
plt.tight_layout()
plt.show()

### 5.5 Correlation Analysis (ML Input Features Only)

In [ ]:
plt.figure(figsize=(8, 6))
corr = df[feature_cols].corr()
sns.heatmap(corr, annot=True, cmap='Blues', fmt='.3f', linewidths=1, square=True, cbar_kws={'label': 'Pearson Correlation'})
plt.title('Correlation Matrix of ML Input Features\n(Performance_Score excluded to prevent distortion)', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

### 5.6 Outlier Inspection

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df[feature_cols], palette=['#2980b9', '#8e44ad', '#16a085', '#d35400'], width=0.5)
plt.title('Outlier Inspection Across All Input Features', fontsize=14, fontweight='bold', pad=15)
plt.ylabel('Score / Percentage (0 - 100)', fontsize=12)
plt.xlabel('Academic Features', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Preprocessing Pipeline Construction

We assemble a Scikit-Learn `Pipeline` containing:
1. `SimpleImputer(strategy='median')`: Imputes missing values using the feature median computed from $X_{train}$.
2. `StandardScaler()`: Standardizes feature scales to $\mu = 0, \sigma = 1$ (vital for Logistic Regression; flexible for tree models).

> **Viva Defense Concept**:
> Tree-based models (Decision Trees, Random Forests) are invariant to monotonic feature scaling because split decisions rely on order rank rather than distance metrics. However, standard scaling is beneficial for gradient-based or distance-based models (Logistic Regression).

In [ ]:
# Construct pipeline
preprocessing_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Fit pipeline STRICTLY on X_train
preprocessing_pipeline.fit(X_train)

# Transform both partitions
X_train_proc = preprocessing_pipeline.transform(X_train)
X_test_proc = preprocessing_pipeline.transform(X_test)

print(f"X_train Transformed Shape: {X_train_proc.shape} | Nulls remaining: {np.isnan(X_train_proc).sum()}")
print(f"X_test  Transformed Shape: {X_test_proc.shape} | Nulls remaining: {np.isnan(X_test_proc).sum()}")
print(f"Transformed Training Feature Means (approx 0.0): {X_train_proc.mean(axis=0).round(4)}")
print(f"Transformed Training Feature Stds  (approx 1.0): {X_train_proc.std(axis=0).round(4)}")

## 7. Phase 2B Summary & Prepared Artifacts

- **Feature Set**: 4 academic features (`MST_Score`, `Quiz_Score`, `Attendance_Percent`, `Assignment_Score`).
- **Target Label**: `Performance_Category` (3 classes: 65.3% Average, 26.1% High, 8.6% Needs Improvement).
- **Excluded**: `Student_ID` (identifier) and `Performance_Score` (target leakage).
- **Missing Values**: Handled via training-fitted median imputation.
- **Preprocessing Pipeline**: Prepared and validated for Phase 3 (Model Training).